In [22]:
import json
from pathlib import Path

countries_path = Path("Resources/Data/countries_export.json")
with open(countries_path) as f:
    countries_data = json.load(f)



In [23]:
# Load flag metadata
flags_path = Path("Resources/Data/flag-icons/country.json")
with open(flags_path) as f:
    flag_data = json.load(f)


In [ ]:

country_lookup = {}


flag_by_iso2 = {item['code']: item for item in flag_data}


for feature in countries_data['features']:
    iso3 = feature['properties'].get('ISO3166-1-Alpha-3')
    iso2 = feature['properties'].get('ISO3166-1-Alpha-2', '').lower()
    name = feature['properties'].get('name')
    
    if iso3:
        flag_info = flag_by_iso2.get(iso2, {})
        country_lookup[iso3] = {
            'name': name,
            'iso2': iso2,
            'flag_path': flag_info.get('flag_4x3', None),
            'feature': feature
        }

print(f"Lookup table built: {len(country_lookup)} countries")


Lookup table built: 237 countries


In [ ]:
# Draw mystery country
from ipyleaflet import Map, GeoJSON
from ipywidgets import Layout
from wdo.geometry.distance import haversine_km
from wdo.games.worldle import choose_target, feature_center, guess_feedback

target = choose_target(countries_data['features'])
print("Mystery country loaded")

# Create map
m = Map(center=(0, 0), zoom=2, layout=Layout(width='100%', height='600px'))

# Style
style = {'color': '#2C3E50', 'fillColor': '#3498DB', 'weight': 2, 'fillOpacity': 0.6}

# Add country
geo = GeoJSON(data=target, style=style, hover_style={'fillOpacity': 0.8})
m.add(geo)


coords = target['geometry']['coordinates']
all_points = []
if target['geometry']['type'] == 'Polygon':
    for ring in coords:
        all_points.extend(ring)
elif target['geometry']['type'] == 'MultiPolygon':
    for polygon in coords:
        for ring in polygon:
            all_points.extend(ring)

lats = [p[1] for p in all_points]
lons = [p[0] for p in all_points]
m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])

m

Mystery country loaded


Map(center=[0, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text'…

In [ ]:

from ipywidgets import Dropdown, Button, VBox, HBox, HTML, Output
from IPython.display import display, clear_output



guesses = []
max_guesses = 6
game_over = False


country_names = sorted([f['properties']['name'] for f in countries_data['features']])
dropdown = Dropdown(options=country_names, description='Guess:')
guess_button = Button(description='Submit Guess', button_style='primary')
history_output = Output()
message_output = Output()

# Handle guess
def on_guess_click(b):
    global game_over
    if game_over:
        return
    
    guess_name = dropdown.value
    guess_feature = [f for f in countries_data['features'] if f['properties']['name'] == guess_name][0]
    
    feedback = guess_feedback(guess_feature, target)
    guesses.append({'name': guess_name, 'feedback': feedback})
    
    with history_output:
        clear_output()
        for g in guesses:
            fb = g['feedback']
            print(f"{g['name']:30s} {fb['arrow']} {fb['distance_km']:7.0f} km  {fb['compass']}")
    
    if feedback['correct']:
        game_over = True
        with message_output:
            clear_output()
            print(f"🎉 Correct! You found {target['properties']['name']} in {len(guesses)} guesses!")
    elif len(guesses) >= max_guesses:
        game_over = True
        with message_output:
            clear_output()
            print(f"Game Over! The country was {target['properties']['name']}")

guess_button.on_click(on_guess_click)

# Layout
display(VBox([
    HBox([dropdown, guess_button]),
    message_output,
    HTML("<h3>Guess History:</h3>"),
    history_output
]))